# 06. Trực quan hoá: insight dataset + SHAP + CIES

Ba phần:
1. **Insight về dataset** (Sparkov, ULB) — chỉ những góc nhìn có tín hiệu thật, kèm 1 giả thuyết phổ biến nhưng **sai** trên dữ liệu này.
2. **SHAP** — model nào "nhìn" feature nào, các model có đồng thuận không.
3. **CIES** — độ ổn định của giải thích qua bootstrap: theo model × kỹ thuật, theo từng feature, và giữa 2 dataset.

Mọi hàm vẽ nằm trong `src/visualization/` (`dataset.py`, `explain.py`); ảnh lưu vào `reports/figures/`.
Phần 3 cần file kết quả của notebook 04/05 (`results/cies_summary_results*.json`) — chưa có thì tự bỏ qua.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, RESULTS_DIR, FIGURES_DIR, TRAIN_FILE, TEST_FILE, TARGET_COL, ULB_TARGET_COL, SEED
from src.visualization import dataset as D
from src.visualization import explain as E

FIG = FIGURES_DIR
FIG.mkdir(parents=True, exist_ok=True)
%matplotlib inline

## 1. Bối cảnh: hai dataset

In [ ]:
sparkov = pd.concat([pd.read_csv(RAW_DATA_DIR / TRAIN_FILE), pd.read_csv(RAW_DATA_DIR / TEST_FILE)], ignore_index=True)
ulb = pd.concat([pd.read_parquet(PROCESSED_DATA_DIR / 'ulb_train.parquet'),
                 pd.read_parquet(PROCESSED_DATA_DIR / 'ulb_test.parquet')], ignore_index=True)
D.plot_imbalance_comparison(sparkov[TARGET_COL], ulb[ULB_TARGET_COL], FIG / 'insight_imbalance.png');

ULB mất cân bằng nặng hơn Sparkov ~3 lần (0.17% so với 0.52%) và chỉ có **492** mẫu fraud tuyệt đối —
đó là lý do các kỹ thuật resampling (SMOTE...) có thể cho kết quả rất khác giữa 2 dataset.

## 2. Sparkov — dữ liệu nói gì

In [ ]:
sp = D.add_sparkov_derived(sparkov)
D.plot_hour_profile(sp, FIG / 'insight_hour.png');

**Giờ đêm (22h–3h) có tỷ lệ fraud gấp ~18 lần ban ngày** dù lưu lượng đêm *thấp hơn*. Lưu ý đường lưu lượng nhảy vọt từ 12h —
Sparkov là dữ liệu **mô phỏng**, nên những hình dạng quá "sạch" như vậy là dấu vết của bộ sinh dữ liệu; đây cũng là lý do cần kiểm chứng trên ULB (dữ liệu thật).

In [ ]:
D.plot_hour_category_heatmap(sp, FIG / 'insight_hour_category.png');

Rủi ro **chồng lên nhau**: `grocery_pos` lúc 22–23h có tới ~39% giao dịch là fraud, `misc_net` ~22%, `shopping_net` ~11%.
Ô trống là các category không có giao dịch ở khung giờ đó — mỗi category chỉ xuất hiện trong vài khung giờ (thêm 1 dấu vết mô phỏng).

In [ ]:
D.plot_amount_fraud_rate(sp, FIG / 'insight_amount.png');

Dưới 200 USD gần như không có fraud (≤0.3%); từ 500 USD trở lên **~21–22% là fraud**. Nhưng dải này chỉ chiếm ~1.2% tổng số giao dịch —
đúng bản chất bài toán: tín hiệu mạnh nhưng nằm ở đuôi hiếm.

In [ ]:
D.plot_distance_myth(sp, FIG / 'insight_distance.png');

**Giả thuyết sai**: "giao dịch xa nơi ở khách thì đáng ngờ". Khoảng cách khách–merchant của fraud và hợp lệ gần như trùng khít (76.3 km so với 76.1 km).
Feature địa lý (`lat`/`long`/`merch_lat`/`merch_long`) vì vậy khó đóng góp khi để nguyên dạng.

In [ ]:
D.plot_velocity(sp, FIG / 'insight_velocity.png');

**Nhịp giao dịch** là tín hiệu thật: khoảng cách giữa 2 giao dịch liên tiếp của cùng 1 thẻ có trung vị **1.4 giờ** với fraud so với **4.4 giờ** với giao dịch hợp lệ.
Pipeline hiện tại **không có** feature kiểu này (chỉ có `hour`, `day_of_week`, `age` và các cột gốc) — 1 hướng feature engineering còn bỏ ngỏ.

In [ ]:
D.plot_month_seasonality(sp, FIG / 'insight_month.png');

Tỷ lệ fraud dao động theo tháng (cao nhất tháng 2 ~0.87%, thấp nhất tháng 12 ~0.30%). Vì train/test của Sparkov tách theo thời gian,
phân phối 2 tập có thể lệch nhau — cần nhớ khi đọc kết quả đánh giá.

## 2b. ULB — feature PCA ẩn danh

In [ ]:
D.plot_ulb_top_features(ulb, ULB_TARGET_COL, save_path=FIG / 'insight_ulb_features.png');

In [ ]:
D.plot_ulb_scatter(ulb, ULB_TARGET_COL, save_path=FIG / 'insight_ulb_scatter.png');

V14, V4, V11, V12 tách fraud khỏi hợp lệ rất rõ (|Cohen's d| ≈ 1.9–2.3). Trong mặt phẳng V14–V4, fraud nằm gọn 1 vùng riêng —
khác Sparkov (tín hiệu rải trên nhiều feature gốc), ULB có vài feature "gánh" gần hết tín hiệu, nên thứ hạng SHAP dễ ổn định hơn.

## 3. SHAP — model nào nhìn feature nào
Train 3 model (Logistic Regression, Random Forest, XGBoost) trên 1 mẫu Sparkov đã encode với `class_weighting`, tính SHAP trên 1 eval set cố định
(100 fraud + 300 hợp lệ). ANN bỏ qua ở đây vì KernelExplainer chậm và chỉ là xấp xỉ. Hyperparameter lấy tự động từ `results/best_params.json`.

In [ ]:
from src.imbalance.resamplers import apply_imbalance
from src.models.train import build_model, train_model
from src.explainability.shap_utils import compute_shap

tr = pd.read_parquet(PROCESSED_DATA_DIR / 'train_encoded.parquet')
te = pd.read_parquet(PROCESSED_DATA_DIR / 'test_encoded.parquet')
feats = [c for c in tr.columns if c != TARGET_COL]

sub = tr.sample(150_000, random_state=SEED)
Xtr, ytr = sub[feats].values, sub[TARGET_COL].values
ev = pd.concat([te[te[TARGET_COL] == 1].sample(100, random_state=SEED),
                te[te[TARGET_COL] == 0].sample(300, random_state=SEED)]).sample(frac=1, random_state=SEED)
Xev = ev[feats].values

shap_by_model, mabs_by_model = {}, {}
for m in ['logistic_regression', 'random_forest', 'xgboost']:
    X_res, y_res, cw = apply_imbalance('class_weighting', Xtr, ytr, seed=SEED)
    mod = train_model(build_model(m, input_dim=Xtr.shape[1], class_weights=cw, seed=SEED), X_res, y_res, model_name=m)
    shap_by_model[m] = compute_shap(mod, m, Xev, X_background=Xtr[:200])
    mabs_by_model[m] = np.abs(shap_by_model[m]).mean(axis=0)
    print(m, 'xong')

In [ ]:
E.plot_shap_beeswarm(shap_by_model['xgboost'], Xev, feats, title='XGBoost — SHAP beeswarm (Sparkov)', save_path=FIG / 'shap_beeswarm_xgboost.png');

In [ ]:
E.plot_shap_bar_by_model(mabs_by_model, feats, top_n=10, save_path=FIG / 'shap_bar_by_model.png');

In [ ]:
E.plot_model_rank_agreement(mabs_by_model, save_path=FIG / 'shap_model_agreement.png');

In [ ]:
top_feat = feats[int(np.argmax(mabs_by_model['xgboost']))]
print('Feature quan trọng nhất (XGBoost):', top_feat)
E.plot_shap_dependence(shap_by_model['xgboost'], Xev, feats, top_feat, save_path=FIG / 'shap_dependence_top.png');

Đọc biểu đồ: **beeswarm** cho thấy chiều tác động (chấm đỏ = giá trị feature cao); **thanh so sánh** cho thấy các model dồn trọng số vào những feature nào;
**ma trận đồng thuận** đo mức các model xếp hạng feature giống nhau (Spearman) — nếu thấp, "feature quan trọng" phụ thuộc vào model bạn chọn.

Trong lần chạy thử: ở 2 model cây, `amt` áp đảo (~37–38% tổng SHAP) và `hour` đứng thứ hai; Random Forest và XGBoost gần như **đồng thuận hoàn toàn** (Spearman 0.95).
Logistic Regression thì **lệch hẳn** (0.55–0.58): nó coi `zip` (~17%) là feature quan trọng nhất, cao hơn cả `amt` (~12%) — 1 mã vùng mà 2 model cây gần như bỏ qua.
Nghĩa là câu trả lời cho "feature nào quan trọng" phụ thuộc vào họ model, không chỉ vào dữ liệu.

## 4. CIES — giải thích có ổn định không
Đọc kết quả từ notebook 04 (Sparkov) và 05 (ULB). Các biểu đồ theo từng run cần `mean_abs_shap` trong `run_logs`
(chỉ có ở kết quả chạy bằng code mới — kết quả cũ vẫn vẽ được heatmap tổng, không vẽ được phần theo run).

In [ ]:
def load_results(name):
    p = RESULTS_DIR / name
    return json.load(open(p, encoding='utf-8')) if p.exists() else None

res_sp = load_results('cies_summary_results.json')
res_ulb = load_results('cies_summary_results_ulb.json')
df_sp = E.cies_results_to_df(res_sp) if res_sp else None
df_ulb = E.cies_results_to_df(res_ulb) if res_ulb else None
print('Sparkov:', 'có' if res_sp else 'CHƯA có (chạy notebook 04)', '| ULB:', 'có' if res_ulb else 'CHƯA có (chạy notebook 05)')

In [ ]:
for name, df in [('sparkov', df_sp), ('ulb', df_ulb)]:
    if df is not None and len(df):
        E.plot_cies_heatmap(df, title=f'CIES score — {name}', save_path=FIG / f'cies_heatmap_{name}.png')
        plt.show()

In [ ]:
def pick(results, best=True):
    ok = [r for r in results if 'cies_metrics' in r and any('mean_abs_shap' in x for x in r.get('run_logs', []))]
    if not ok:
        return None
    return (max if best else min)(ok, key=lambda r: r['cies_metrics']['cies_score'])

for name, res in [('sparkov', res_sp), ('ulb', res_ulb)]:
    if not res:
        continue
    for tag, best in [('ổn định nhất', True), ('kém ổn định nhất', False)]:
        r = pick(res, best)
        if r is None:
            print(name, ': kết quả cũ, không có mean_abs_shap theo run — chạy lại CIES bằng code mới để vẽ phần này.')
            break
        print(f'=== {name} — tổ hợp {tag}: {r["combination"]} (CIES {r["cies_metrics"]["cies_score"]:.3f}) ===')
        key = 'best' if best else 'worst'
        E.plot_rank_stability(r, save_path=FIG / f'cies_rank_stability_{name}_{key}.png'); plt.show()
        E.plot_topk_frequency(r, save_path=FIG / f'cies_topk_{name}_{key}.png'); plt.show()
        E.plot_pairwise_spearman(r, save_path=FIG / f'cies_spearman_{name}_{key}.png'); plt.show()

Đọc biểu đồ: **hộp thứ hạng hẹp** = feature giữ nguyên vị trí qua mọi bootstrap; **top-k frequency < 100%** = ranh giới top-k không chắc chắn;
**ma trận Spearman có ô nhạt** = có cặp run cho giải thích khác hẳn nhau dù cùng model, cùng kỹ thuật, chỉ khác mẫu bootstrap.

In [ ]:
if df_sp is not None and df_ulb is not None and len(df_sp) and len(df_ulb):
    try:
        E.plot_cies_dataset_compare(df_sp, df_ulb, 'Sparkov', 'ULB', save_path=FIG / 'cies_sparkov_vs_ulb.png')
        plt.show()
    except ValueError as e:
        print(e)
else:
    print('Cần cả 2 file kết quả để so sánh Sparkov với ULB.')

In [ ]:
bench = RESULTS_DIR / 'model_benchmark_results.csv'
if df_sp is not None and len(df_sp) and bench.exists():
    E.plot_cies_vs_prauc(df_sp, pd.read_csv(bench), save_path=FIG / 'cies_vs_prauc.png')
    plt.show()
else:
    print('Cần cies_summary_results.json (notebook 04) và model_benchmark_results.csv (notebook 03).')